# Day 2: Bonds and FX in depth
Date: 2 October 2026 Goal: price a real bond with many payments, understand why long bonds are riskier, and measure that risk in basis points.

The numbers in this notebook are simple examples to explain the ideas, not real market data.

## 1. Pricing a bond with many payments

On Day 1 I priced a bond that pays once, in one year: price = 1050 ÷ (1 + yield).
A real bond pays a coupon **every year**, and the face value at the end. Each payment arrives at a different time, so each one is **discounted** separately, then I add them up:

**Price = C ÷ (1+y)¹ + C ÷ (1+y)² + … + (C + F) ÷ (1+y)ⁿ**

- C = coupon payment, F = face value, y = yield, n = years to maturity

In words: **a bond's price is the value today of all the money it will pay me.**

In [1]:
def bond_price(face_value, coupon_rate, years, ytm):
    """Price of a bond paying an annual coupon, discounted at the yield to maturity (ytm)."""
    coupon = face_value * coupon_rate
    price = 0
    for t in range(1, years + 1):
        price += coupon / (1 + ytm) ** t          # each coupon, discounted
    price += face_value / (1 + ytm) ** years      # face value, paid at the end
    return price

# A 5-year bond, €1,000 face value, 5% coupon
for ytm in [0.04, 0.05, 0.06]:
    print(f"Yield {ytm * 100:.0f}%  ->  price €{bond_price(1000, 0.05, 5, ytm):.2f}")

Yield 4%  ->  price €1044.52
Yield 5%  ->  price €1000.00
Yield 6%  ->  price €957.88


### My questions and answers

**Q: Why do I discount each coupon separately?**
Because money received later is worth less today. €50 in one year is worth more to me than €50 in five years, so the five-year coupon is divided by a bigger number: (1 + y)⁵.

**Q: Does this match what I found on Day 1?**
Yes. At a 5% yield the price is exactly €1,000 (yield = coupon). When the yield goes up to 6%, the price falls to about €957.88. When it goes down to 4%, the price rises to about €1,044.52. Same rule: price and yield move in opposite directions.

**Q: Why write a function instead of just calculating?**
Because I can reuse it for any bond, and I can **test** it. For example, I know that when yield = coupon, the price must equal the face value. If my function doesn't give €1,000 there, it has a bug.

In [2]:
# A simple test: when yield = coupon rate, the price must equal face value
price = bond_price(1000, 0.05, 5, 0.05)
assert abs(price - 1000) < 1e-9, f"Expected 1000, got {price}"
print("Test passed: par bond prices at face value")

Test passed: par bond prices at face value


## 2. Par, premium and discount bonds

| Bond trades at… | When | Price vs face value |
|---|---|---|
| **Par** | yield = coupon | price = face value |
| **Premium** | yield < coupon | price > face value |
| **Discount** | yield > coupon | price < face value |

### My questions and answers

**Q: Why would anyone pay more than €1,000 for a bond that only pays back €1,000?**
Because its coupon is higher than what new bonds pay. If market yields are 4% and my bond pays 5%, buyers pay extra for those bigger coupons. They lose the extra at maturity but earn it back through the coupons. In the end, their return is about 4%, like the market.

**Q: What happens to the price as maturity gets closer?**
It moves towards the face value. On the last day, the bond is worth exactly what it pays back.

## 3. Why long bonds are riskier: DV01 and duration

If yields rise by the same amount, a **long** bond loses more value than a short one. That's because more of its payments are far in the future, and those are hit hardest by discounting.

Banks measure this with two numbers:

- **DV01** ("dollar value of one basis point"): how much the price changes when the yield moves by **1 bp**. If DV01 = €0.43, a 1 bp rise in yield makes the bond lose about €0.43.
- **Duration**: the average time (in years) until I receive the bond's money, weighted by how much each payment is worth today. Longer duration = more sensitive to rates.

In [3]:
def dv01(face_value, coupon_rate, years, ytm):
    """Price change when the yield rises by 1 basis point (0.0001)."""
    return bond_price(face_value, coupon_rate, years, ytm) - bond_price(face_value, coupon_rate, years, ytm + 0.0001)

def macaulay_duration(face_value, coupon_rate, years, ytm):
    """Average time to receive the bond's cash flows, weighted by their present value."""
    coupon = face_value * coupon_rate
    price = bond_price(face_value, coupon_rate, years, ytm)
    weighted_time = 0
    for t in range(1, years + 1):
        cash_flow = coupon + (face_value if t == years else 0)
        weighted_time += t * cash_flow / (1 + ytm) ** t
    return weighted_time / price

print(f"{'Maturity':>9} | {'Duration (yrs)':>14} | {'DV01 (€)':>8}")
for years in [2, 5, 10, 30]:
    d = macaulay_duration(1000, 0.05, years, 0.05)
    risk = dv01(1000, 0.05, years, 0.05)
    print(f"{years:>9} | {d:>14.2f} | {risk:>8.2f}")

 Maturity | Duration (yrs) | DV01 (€)
        2 |           1.95 |     0.19
        5 |           4.55 |     0.43
       10 |           8.11 |     0.77
       30 |          16.14 |     1.54


### My questions and answers

**Q: What does this table tell me?**
The 30-year bond's DV01 is about 8 times bigger than the 2-year bond's. If rates rise by the same amount, the long bond loses much more money. Duration grows with maturity too, but it's always shorter than the maturity, because the coupons pay part of my money back early.

**Q: Why is duration shorter than maturity?**
Because I don't wait until the end for all my money. The coupons arrive every year, so the "average" waiting time is less than the full maturity. A bond with no coupons (a zero-coupon bond) has duration = maturity.

**Q: Why do banks care about DV01?**
Because it turns rate risk into money. A trader can say: "My portfolio loses €50,000 for every 1 bp rise in rates." Risk systems like Murex calculate this for whole portfolios, and test engineers check that these numbers are correct.

## 4. Cross rates

Most currencies are quoted against the US dollar (EUR/USD, GBP/USD, USD/JPY). But a client might want **EUR/GBP**, which isn't quoted against the dollar. So it's calculated from two dollar rates. This is called a **cross rate**.

Two rules:
- If USD is the **quote** currency in both rates → **divide**. EUR/GBP = EUR/USD ÷ GBP/USD
- If USD is the **quote** in one and the **base** in the other → **multiply**. EUR/JPY = EUR/USD × USD/JPY

Trick to check: write the rates as fractions and cancel the USD.
EUR/USD × USD/JPY → (USD per EUR) × (JPY per USD) = JPY per EUR = EUR/JPY ✅

In [4]:
eur_usd = 1.10   # 1 EUR = 1.10 USD
gbp_usd = 1.28   # 1 GBP = 1.28 USD
usd_jpy = 150.0  # 1 USD = 150 JPY

eur_gbp = eur_usd / gbp_usd
eur_jpy = eur_usd * usd_jpy

print(f"EUR/GBP = {eur_gbp:.4f}")
print(f"EUR/JPY = {eur_jpy:.2f}")

# Test: going EUR -> GBP -> USD must give the same result as EUR -> USD directly
assert abs(eur_gbp * gbp_usd - eur_usd) < 1e-12
print("Test passed: cross rate is consistent")

EUR/GBP = 0.8594
EUR/JPY = 165.00
Test passed: cross rate is consistent


### My questions and answers

**Q: Why not just quote EUR/GBP directly?**
Big pairs like EUR/GBP are quoted directly too. But for less common pairs (for example, a currency in Asia against one in South America), banks build the rate from two dollar rates, because the dollar is on the other side of most trades.

**Q: How do I know whether to multiply or divide?**
I write the units and check that USD cancels out. If it doesn't cancel, I picked the wrong operation.

## 5. FX forwards

A **spot** trade is exchanged now (in practice, in 2 business days).
A **forward** fixes today the rate for an exchange on a **future date**. A company that will receive dollars in one year can lock in the rate now, so it isn't hurt if the dollar falls.

The forward rate isn't a guess about the future. It comes from the **interest rates** of the two currencies:

**Forward = Spot × (1 + r_quote)ᵀ ÷ (1 + r_base)ᵀ**

This is called **interest rate parity**. If it didn't hold, someone could borrow in one currency, invest in the other, lock the forward, and make a risk-free profit (an **arbitrage**).

The difference between forward and spot, in pips, is called the **forward points**.

In [5]:
spot = 1.10      # EUR/USD
r_usd = 0.04     # USD interest rate (quote currency)
r_eur = 0.02     # EUR interest rate (base currency)
T = 1            # years

forward = spot * (1 + r_usd) ** T / (1 + r_eur) ** T
forward_points = (forward - spot) * 10000

print(f"Spot EUR/USD:      {spot:.4f}")
print(f"1-year forward:    {forward:.4f}")
print(f"Forward points:    {forward_points:+.0f} pips")

Spot EUR/USD:      1.1000
1-year forward:    1.1216
Forward points:    +216 pips


### My questions and answers

**Q: Why is the forward higher than the spot here?**
Because USD interest rates (4%) are higher than EUR rates (2%). The currency with the **higher** interest rate is cheaper in the forward market. So in one year, one euro buys more dollars. If it were the other way, people could earn the extra 2% in dollars **and** get a good forward rate. That's free money, which markets don't allow.

**Q: Does the forward predict where the rate will be in a year?**
No. It's just what the interest rates imply today. The real spot rate in one year can be very different.

**Q: Who uses forwards?**
Companies that know they'll pay or receive foreign currency later (importers, exporters), and banks managing their risk. Forwards are also the first **derivative** I'll study on Day 3.

## 6. Test yourself

<details><summary>1. A 10-year bond has a 4% coupon and the market yield is 5%. Does it trade at par, a premium or a discount?</summary>
A discount. Yield > coupon, so price < face value.
</details>

<details><summary>2. Which has a bigger DV01: a 2-year bond or a 10-year bond (same coupon and yield)?</summary>
The 10-year bond. Longer maturity = more sensitive to rates.
</details>

<details><summary>3. A bond has a DV01 of €0.77. Yields rise by 10 bp. How much does it lose, roughly?</summary>
About €7.70 (0.77 × 10).
</details>

<details><summary>4. EUR/USD = 1.08 and USD/JPY = 155. What is EUR/JPY?</summary>
1.08 × 155 = 167.40 (multiply: USD cancels).
</details>

<details><summary>5. EUR rates are 3% and GBP rates are 5%. Is the EUR/GBP forward above or below spot?</summary>
Above. GBP (the quote currency) has the higher rate, so Forward = Spot × 1.05 ÷ 1.03 > Spot.
</details>

<details><summary>6. What is the duration of a 5-year zero-coupon bond?</summary>
5 years. All the money comes at the end, so duration = maturity.
</details>

## What I learned

- A bond's price is the value today of **all** its future payments, each one discounted separately.
- A bond trades at **par** when yield = coupon, at a **premium** when yield < coupon, and at a **discount** when yield > coupon.
- Long bonds are riskier: **DV01** tells me how much money I lose per 1 bp rise, and **duration** tells me how long, on average, my money is tied up.
- Writing pricing as **functions with tests** lets me check them automatically. For example, a par bond must price at face value.
- **Cross rates** are built from two dollar rates. I multiply or divide so that USD cancels out.
- An **FX forward** comes from interest rates, not from predictions. The higher-rate currency is cheaper in the forward market, otherwise there would be an arbitrage.